# Rapport — Hackathon #26
## Le climat en France de 1900 à 2100 : données, modèles et projections

Données : Météo France, NOAA, CITEPA, PSMSL, INSEE, EM-DAT  
Modèle : Prophet (Meta)  
Suivi des expériences : MLflow

---

**Sommaire**

1. Pourquoi ce sujet ?
2. Les données qu'on a utilisées
3. Ce que l'histoire nous dit
4. Comment on a fait les prédictions
5. Est-ce que les modèles sont bons ?
6. Ce qui nous attend en 2030, 2050, 2100
7. Que peut-on faire concrètement ?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.facecolor'] = '#0e1117'
plt.rcParams['axes.facecolor']   = '#1a1d27'
plt.rcParams['text.color']       = 'white'
plt.rcParams['axes.labelcolor']  = 'white'
plt.rcParams['xtick.color']      = 'white'
plt.rcParams['ytick.color']      = 'white'
plt.rcParams['axes.edgecolor']   = '#444'
plt.rcParams['grid.color']       = '#333'
plt.rcParams['grid.linestyle']   = '--'
plt.rcParams['grid.alpha']       = 0.5

df = pd.read_csv('data/clean/dataset_final.csv')
df['annee'] = pd.to_numeric(df['annee'], errors='coerce')

print(f'{df.shape[0]} années dans le dataset, de {int(df.annee.min())} à {int(df.annee.max())}')
df.tail(5)

---
## 1. Pourquoi ce sujet ?

On entend beaucoup parler du changement climatique, mais rarement avec des chiffres précis sur la France. L'idée de ce projet c'est de partir de données officielles — pas d'articles d'opinion — et de répondre à une question simple : **est-ce qu'on voit vraiment quelque chose dans les données, et si oui, où ça nous mène ?**

On a donc construit un pipeline qui va chercher les données à la source (Météo France, NOAA, PSMSL…), les nettoie, les assemble, puis entraîne des modèles pour faire des projections jusqu'en 2100 selon trois scénarios.

Périmètre : France métropolitaine, 1900 à 2025 pour l'historique, 2026 à 2100 pour les projections.

---
## 2. Les données qu'on a utilisées

On a rassemblé neuf indicateurs provenant de sources officielles :

- **Température moyenne annuelle (°C)** — Météo France, 1851 à 2025
- **Jours chauds au-dessus de 30°C** — Météo France, 1950 à 2025
- **Jours de gel en-dessous de 0°C** — Météo France, 1950 à 2025
- **CO₂ atmosphérique (ppm)** — NOAA Mauna Loa, 1958 à 2025
- **Émissions de GES par secteur (Mt CO₂eq)** — CITEPA, 1960 à 2023
- **Niveau de la mer (anomalie en mm)** — PSMSL, 36 ports français, 1900 à 2023
- **Empreinte carbone par habitant (t/an)** — INSEE, 1990 à 2024
- **Date des vendanges (jour de l'année)** — estimations régionales, 1900 à 2026
- **Coût des catastrophes naturelles (Mrd USD)** — EM-DAT, 1950 à 2023

On a choisi ces indicateurs parce qu'ils ont des séries longues, qu'ils couvrent plusieurs dimensions du problème (physique, économique, agricole) et qu'ils correspondent aux axes du plan national d'adaptation au changement climatique (PNACC-3).



---
## 3. Ce que l'histoire nous dit

In [ ]:
# Température annuelle + tendance lissée sur 10 ans
df_temp = df[['annee', 'temp_moy_france']].dropna()
df_temp = df_temp[df_temp['temp_moy_france'] > 9.5]  # on retire les années incomplètes

baseline = df_temp[df_temp['annee'] <= 1920]['temp_moy_france'].mean()
moy10    = df_temp.set_index('annee')['temp_moy_france'].rolling(10, center=True).mean()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df_temp['annee'], df_temp['temp_moy_france'],
        color='#aaaaaa', lw=0.8, alpha=0.7, label='Mesure annuelle')
ax.plot(moy10.index, moy10.values,
        color='#fb923c', lw=2, label='Tendance (moyenne glissante 10 ans)')
ax.axhline(baseline, color='#60a5fa', lw=1.5, ls='--',
           label=f'Niveau de référence 1900-1920 ({baseline:.1f}°C)')

derniere_temp = df_temp['temp_moy_france'].iloc[-1]
ax.annotate(f'+{derniere_temp - baseline:.2f}°C\ndepuis 1900',
            xy=(df_temp['annee'].iloc[-1], derniere_temp),
            xytext=(1990, baseline + 0.5),
            arrowprops=dict(arrowstyle='->', color='white'),
            color='white', fontsize=10)

ax.set_title('Température moyenne annuelle en France (1900-2025)', fontsize=13, pad=12)
ax.set_xlabel('Année'); ax.set_ylabel('°C')
ax.legend(); ax.grid(True)
plt.tight_layout()
plt.savefig('data/resultats/fig_temp_historique.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'La France a gagné +{derniere_temp - baseline:.2f}°C depuis le début du XXe siècle.')

In [ ]:
# Tous les indicateurs en un seul coup d'oeil
indicateurs = {
    'CO2 (ppm)':                 'co2_ppm',
    'Niveau mer (mm)':            'niveau_mer_mm',
    'Jours chauds >= 30C':        'jours_chauds_30',
    'Jours de gel <= 0C':         'jours_gel',
    'Empreinte (t CO2/hab)':      'empreinte_tCO2_hab',
    'Vendanges (jour de annee)':  'jour_vendanges',
    'Cout catastrophes (Mrd$)':   'dommages_Mrd_USD',
}
indicateurs = {k: v for k, v in indicateurs.items() if v in df.columns}

n = len(indicateurs)
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
couleurs = ['#fb923c', '#60a5fa', '#f87171', '#4ade80', '#c084fc', '#a78bfa', '#fbbf24']

for i, (label, col) in enumerate(indicateurs.items()):
    ax = axes[i]
    sub = df[['annee', col]].dropna()
    ax.plot(sub['annee'], sub[col], color=couleurs[i], lw=1.5)
    ax.set_title(label, fontsize=10)
    ax.set_xlabel('Annee', fontsize=8)
    ax.tick_params(labelsize=8)
    ax.grid(True, alpha=0.4)

for j in range(n, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Tous les indicateurs en un coup d\'oeil (France, 1900-2025)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('data/resultats/fig_indicateurs_overview.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Corrélations entre indicateurs
cols_corr = ['temp_moy_france', 'co2_ppm', 'niveau_mer_mm',
             'jours_chauds_30', 'empreinte_tCO2_hab', 'jour_vendanges']
cols_corr = [c for c in cols_corr if c in df.columns]

df_corr = df[cols_corr].dropna()
corr    = df_corr.corr()

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr, cmap='RdYlGn', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_xticks(range(len(cols_corr))); ax.set_xticklabels(cols_corr, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(len(cols_corr))); ax.set_yticklabels(cols_corr, fontsize=9)

for i in range(len(cols_corr)):
    for j in range(len(cols_corr)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}',
                ha='center', va='center', fontsize=8,
                color='black' if abs(corr.iloc[i, j]) < 0.6 else 'white')

ax.set_title('Plus la valeur est proche de 1, plus les deux indicateurs evoluent ensemble', fontsize=10)
plt.tight_layout()
plt.savefig('data/resultats/fig_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

print('Lien avec la temperature (1 = evoluent ensemble, 0 = independants) :')
print(corr['temp_moy_france'].drop('temp_moy_france').sort_values(ascending=False).round(2).to_string())

---
## 4. Comment on a fait les prédictions

### Le modèle : Prophet

On a utilisé **Prophet**, un outil open-source développé par Meta en 2017 pour faire des prévisions sur des séries temporelles. C'est adapté à notre cas pour plusieurs raisons :

- Il gère bien les longues séries avec des tendances qui changent au fil du temps
- Il est robuste aux données manquantes (pas besoin de combler les trous)
- Il donne une **fourchette d'incertitude** en plus de la valeur prédite
- On peut voir la tendance séparément du bruit

L'idée de base c'est de décomposer la série en trois parties :

$$y(t) = \text{tendance}(t) + \text{saisonnalité}(t) + \text{bruit}(t)$$

Avec des données annuelles comme les nôtres, la saisonnalité n'a pas de sens (un seul point par an), donc Prophet ne modélise que la tendance et le bruit.

### Comment on vérifie que le modèle est bon

On utilise une validation croisée temporelle : on entraîne le modèle sur le passé et on prédit le futur proche, puis on recommence en avançant dans le temps. C'est l'équivalent de cacher la fin du livre pour voir si on devine bien l'histoire.

On mesure trois choses :
- **RMSE** : erreur moyenne, mais les grosses erreurs comptent double
- **MAE** : erreur moyenne simple, plus intuitive
- **MAPE** : erreur en pourcentage — attention, elle devient absurde si les valeurs sont proches de zéro

### Les trois scénarios

Pour la température, on a construit trois trajectoires inspirées du GIEC :

- **Optimiste** : +1.4°C en 2100 — on fait vraiment beaucoup d'efforts (SSP1-1.9)
- **Intermédiaire** : +2.7°C en 2100 — on continue à peu près comme maintenant (SSP2-4.5)
- **Pessimiste** : +4.4°C en 2100 — on ne change rien du tout (SSP5-8.5)

---
## 5. Est-ce que les modèles sont bons ?

In [ ]:
import os

path_perf = 'data/resultats/comparaison_performances.csv'
if os.path.exists(path_perf):
    df_perf = pd.read_csv(path_perf)
    print('Performances des modeles (plus les valeurs sont basses, mieux le modele predit) :')
    display(df_perf.style
            .format({'rmse': '{:.4f}', 'mae': '{:.4f}', 'mape': '{:.2f}'}, na_rep='—')
            .background_gradient(subset=['rmse', 'mae'], cmap='RdYlGn_r')
            .set_caption('RMSE et MAE sont dans l\'unite de la variable. MAPE est en %.'))
    print()
    print('Deux modeles sans resultat :')
    print('  - jours_chauds : serie trop courte pour valider correctement')
    print('  - cout_catastrophes : donnees trop irregulieres (Prophet ne s\'y adapte pas bien)')
    print()
    print('Le MAPE du niveau de la mer (108%) ne veut rien dire :')
    print('les valeurs sont centrees autour de 0, ce qui fausse le calcul en pourcentage.')
    print('On regarde le RMSE a la place : 30 mm, soit environ 3 cm d\'erreur en moyenne.')
else:
    print('Lancer modele_prophet.py pour generer les performances.')

In [ ]:
# Observations + projection Prophet + 3 scenarios
fc_path = 'data/resultats/forecast_temperature.csv'
if os.path.exists(fc_path):
    fc_temp = pd.read_csv(fc_path)
    fc_temp['annee'] = pd.to_numeric(fc_temp['annee'], errors='coerce')

    fig, ax = plt.subplots(figsize=(14, 5))

    df_t = df[['annee', 'temp_moy_france']].dropna()
    df_t = df_t[df_t['temp_moy_france'] > 9.5]
    ax.plot(df_t['annee'], df_t['temp_moy_france'],
            'white', lw=1.5, label='Ce qu\'on a mesure', zorder=5)

    fc_fut = fc_temp[fc_temp['annee'] > df_t['annee'].max()]
    ax.fill_between(fc_fut['annee'], fc_fut['yhat_lower'], fc_fut['yhat_upper'],
                    alpha=0.2, color='steelblue', label='Fourchette de projection (95%)')
    ax.plot(fc_fut['annee'], fc_fut['yhat'],
            color='#60a5fa', lw=2, ls='--', label='Si la tendance continue (Prophet)')

    sc_path = 'data/resultats/scenarios_temperature.csv'
    if os.path.exists(sc_path):
        df_sc = pd.read_csv(sc_path)
        annee_ref = int(df_t['annee'].iloc[-1])
        temp_ref  = df_t['temp_moy_france'].iloc[-1]
        for sc, coul, label in [
            ('optimiste',     '#4ade80', 'Scenario optimiste (+1.4C)'),
            ('intermediaire', '#fb923c', 'Scenario intermediaire (+2.7C)'),
            ('pessimiste',    '#f87171', 'Scenario pessimiste (+4.4C)'),
        ]:
            rows = df_sc[df_sc['scenario'] == sc]
            xs = [annee_ref] + rows[rows['annee'] > annee_ref]['annee'].tolist()
            ys = [temp_ref]  + rows[rows['annee'] > annee_ref]['temp_proj_C'].tolist()
            ax.plot(xs, ys, color=coul, lw=2.5, marker='o', ms=7, label=label)

    ax.set_title('Temperature en France : passe et projections selon trois scenarios', fontsize=13)
    ax.set_xlabel('Annee'); ax.set_ylabel('C')
    ax.legend(loc='upper left', fontsize=9); ax.grid(True)
    plt.tight_layout()
    plt.savefig('data/resultats/fig_forecast_scenarios.png', dpi=150, bbox_inches='tight')
    plt.show()

---
## 6. Ce qui nous attend en 2030, 2050, 2100

In [ ]:
# Projections de temperature par scenario
sc_path = 'data/resultats/scenarios_temperature.csv'
if os.path.exists(sc_path):
    df_sc = pd.read_csv(sc_path)
    pivot = df_sc.pivot(index='annee', columns='scenario',
                        values=['temp_proj_C', 'anomalie_C'])
    pivot.columns = [f'{b} - {a}' for a, b in pivot.columns]
    print('Temperature prevue en Celsius selon le scenario :')
    display(pivot.round(2))

In [ ]:
# Projections des autres indicateurs
resultats_proj = {}
for annee_cible in [2030, 2050, 2100]:
    row = {}
    for nom, label in [
        ('co2',               'CO2 (ppm)'),
        ('niveau_mer',        'Niveau de la mer (mm)'),
        ('empreinte_carbone', 'Empreinte carbone (t/hab)'),
        ('vendanges',         'Vendanges (jour de annee)'),
    ]:
        path = f'data/resultats/forecast_{nom}.csv'
        if os.path.exists(path):
            fc = pd.read_csv(path)
            val = fc[fc['annee'] == annee_cible]['yhat']
            row[label] = round(val.values[0], 1) if not val.empty else None
    resultats_proj[annee_cible] = row

df_proj = pd.DataFrame(resultats_proj).T
df_proj.index.name = 'Horizon'
print('Projections si la tendance actuelle continue :')
display(df_proj)

---
## 7. Que peut-on faire concrètement ?

### Ce qu'on voit dans les données

Quatre signaux ressortent clairement :

- Plus de 2.6°C de hausse depuis 1900, et le nombre de jours chauds augmente — davantage de canicules dès 2030
- Les vendanges arrivent de plus en plus tôt — signe d'un stress hydrique et d'un dérèglement agricole déjà en cours
- Le niveau de la mer a monté de 120 mm en moyenne sur les ports français depuis 1961 — risques de submersion à partir de 2050
- Le CO₂ monte sans s'arrêter depuis 1958 — c'est la cause directe du réchauffement

### Ce que chacun peut faire

**Comme citoyen :**
- Réduire son empreinte carbone sous 5 t CO₂/an (on est à environ 9 t en moyenne en France)
- Isoler son logement — c'est souvent le levier le plus efficace
- Manger moins de viande rouge

**Comme élu local :**
- Intégrer les projections climatiques dans les documents d'urbanisme
- Végétaliser les villes pour limiter les îlots de chaleur
- Réviser les plans de prévention des risques naturels

**Comme entreprise :**
- Faire un bilan carbone et se fixer une trajectoire de réduction
- Anticiper les risques climatiques dans les plans de continuité d'activité

### Le lien avec les engagements officiels

- **Accord de Paris** : objectif +1.5°C — on est déjà à +2.6°C en France
- **PNACC-3 (2024)** : plan national d'adaptation — nos indicateurs couvrent tous leurs axes de risque
- **SNBC** : neutralité carbone en 2050 — l'empreinte individuelle doit être divisée par environ 4

---

## Annexe : comment reproduire ces résultats

Le projet fonctionne en trois étapes :

1. `pipeline.py` — récupère et nettoie les données, produit `data/clean/dataset_final.csv`
2. `modele_prophet.py` — entraîne les modèles, produit les forecasts dans `data/resultats/`, trace tout dans MLflow
3. `app.py` — dashboard Streamlit qui lit les résultats

Pour relancer depuis zéro :
```bash
pip install pandas requests openpyxl prophet matplotlib mlflow streamlit plotly folium streamlit-folium
python pipeline.py
python modele_prophet.py
python -m streamlit run app.py
```

In [ ]:
# Toutes les runs MLflow enregistrees pendant la modelisation
import mlflow

try:
    client = mlflow.tracking.MlflowClient()
    exp    = client.get_experiment_by_name('hackathon26_climat_france')
    if exp:
        runs = client.search_runs(exp.experiment_id, order_by=['metrics.rmse ASC'])
        rows = []
        for r in runs:
            rows.append({
                'modele':    r.data.tags.get('mlflow.runName', r.info.run_id),
                'RMSE':      r.data.metrics.get('rmse'),
                'MAE':       r.data.metrics.get('mae'),
                'MAPE %':    r.data.metrics.get('mape'),
                'nb points': r.data.params.get('n_observations'),
                'periode':   f"{r.data.params.get('annee_debut')}-{r.data.params.get('annee_fin')}"
            })
        df_mlflow = pd.DataFrame(rows)
        print(f'{len(runs)} runs enregistrees dans MLflow :')
        display(df_mlflow)
    else:
        print('Pas de runs trouvees. Lancer modele_prophet.py.')
except Exception as e:
    print(f'MLflow non accessible : {e}')